# POC — Cross-sectional return regression

This notebook preserves the **first return-regression proof of concept** for the current
canonical feature set. Its purpose is historical and diagnostic: establish what Ridge and a
first-pass XGBoost configuration can learn from 5-, 10-, and 15-session forward returns before
the project moves to the dedicated short-horizon comparison notebook.

The notebook deliberately:

- reuses the repository's canonical dataset and temporal-split contracts;
- evaluates only the outer validation period and never reads the locked test period;
- treats stock selection as a daily cross-sectional problem;
- reports prediction cardinality and score ties explicitly;
- treats dates with one common score as **no-signal dates**;
- keeps expensive bootstrap and permutation analysis opt-in and progress-visible.

This is a frozen baseline, not the place for new horizons, ATR-normalized labels, rankers, or
neural-network experiments. Those belong in the next notebooks.

## 1. Configuration

The XGBoost parameters below reproduce the original first-pass experiment. They are intentionally
retained here even though later diagnostics indicate that the model is heavily regularized and
emits very coarse predictions. The next notebook will use regression-specific alternatives.

Set `TICKERS` to a small tuple for a smoke test. Leave it as `None` to discover all locally stored
tickers for the configured provider.

In [ ]:
from datetime import date
from pathlib import Path

PROVIDER = "yfinance"

# Example smoke test: ("AAK.ST", "SAAB-B.ST", "VOLV-B.ST")
TICKERS: tuple[str, ...] | None = None

DATA_CUTOFF = date(2026, 7, 24)

TRAIN_START = date(2000, 1, 1)
TRAIN_END = date(2022, 12, 31)
VALIDATION_START = date(2023, 1, 1)
VALIDATION_END = date(2024, 12, 31)
TEST_START = date(2025, 1, 1)
TEST_END = DATA_CUTOFF

HORIZONS = (5, 10, 15)
TOP_K_VALUES = (1, 3, 5)
RANDOM_SEED = 42

# Resampling is intentionally opt-in because the original implementation was slow.
RUN_RESAMPLING = False
BOOTSTRAP_ITERATIONS = 500
PERMUTATION_ITERATIONS = 250

# Historical first-pass parameters retained for reproducibility.
XGB_PARAMS = {
    "n_estimators": 120,
    "learning_rate": 0.03,
    "max_depth": 4,
    "min_child_weight": 5,
    "gamma": 0.2,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "reg_alpha": 0.1,
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "tree_method": "hist",
    "n_jobs": -1,
    "random_state": RANDOM_SEED,
}

## 2. Build the canonical dataset and fixed outer split

`ExperimentSpec` still requires a model identity, but model fitting remains notebook-local. The
production classification model type is therefore used only as a stable placeholder identity.

In [ ]:
import pandas as pd
from sqlalchemy import text

from swingtrader.data.db import resolve_database_engine
from swingtrader.data.features import DEFAULT_FEATURE_SET
from swingtrader.modeling.datasets import (
    UniverseSpec,
    FORWARD_RETURN_PRIMARY_TASK,
    FORWARD_RETURN_TARGET_SET,
    build_temporal_dataset,
)
from swingtrader.modeling.experiments import (
    ExperimentSpec,
    FixedTemporalSplitter,
    ModelSpec,
    TemporalSplitSpec,
)
from swingtrader.modeling.training import LOGISTIC_REGRESSION_MODEL_TYPE


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current directory.")


repo_root = find_repo_root(Path.cwd())
database_path = repo_root / "data" / "swingtrader.sqlite"
database_url = f"sqlite+pysqlite:///{database_path.as_posix()}"
engine = resolve_database_engine(database_url=database_url)

if TICKERS is None:
    with engine.connect() as connection:
        ticker_rows = connection.execute(
            text(
                """
                SELECT DISTINCT ticker
                FROM bronze_market_daily_prices
                WHERE provider = :provider
                ORDER BY ticker
                """
            ),
            {"provider": PROVIDER},
        ).fetchall()
    resolved_tickers = tuple(row[0] for row in ticker_rows)
else:
    resolved_tickers = tuple(TICKERS)

if not resolved_tickers:
    raise RuntimeError(
        f"No bronze tickers were found for provider {PROVIDER!r}. "
        "Populate the local database or set TICKERS explicitly."
    )

{
    "database_path": database_path,
    "ticker_count": len(resolved_tickers),
    "first_tickers": resolved_tickers[:10],
}

In [ ]:
feature_set = DEFAULT_FEATURE_SET

placeholder_model = ModelSpec(
    name="notebook_regression_poc",
    version="1",
    model_type=LOGISTIC_REGRESSION_MODEL_TYPE,
    hyperparameters={},
    feature_columns=None,
)

universe = UniverseSpec(
    name="local_bronze_regression_universe",
    version="1",
    provider=PROVIDER,
    tickers=resolved_tickers,
)

split_spec = TemporalSplitSpec(
    name="regression_poc_fixed_holdout",
    version="1",
    train_start=TRAIN_START,
    train_end=TRAIN_END,
    validation_start=VALIDATION_START,
    validation_end=VALIDATION_END,
    test_start=TEST_START,
    test_end=TEST_END,
)

experiment_spec = ExperimentSpec(
    name="return_regression_poc",
    version="1",
    feature_set=feature_set,
    target_set=FORWARD_RETURN_TARGET_SET,
    task=FORWARD_RETURN_PRIMARY_TASK,
    universe=universe,
    data_start=TRAIN_START,
    data_end=TEST_END,
    split=split_spec,
    model=placeholder_model,
    random_seeds={"model": RANDOM_SEED, "evaluation": RANDOM_SEED + 1},
)

bundle = build_temporal_dataset(engine=engine, spec=experiment_spec.dataset_spec)
split_result = FixedTemporalSplitter(experiment_spec.split).assign(bundle)

{
    "experiment_digest": experiment_spec.digest,
    "ticker_count": len(resolved_tickers),
    "generated_feature_count": len(bundle.manifest.feature_columns),
    "target_columns": bundle.manifest.target_columns,
    "outer_train": split_result.summary("train").to_manifest(),
    "outer_validation": split_result.summary("validation").to_manifest(),
}

## 3. Extract aligned train and validation frames

The canonical `trading_date` remains an index level. The helper below keeps features, labels, and
sample metadata aligned after removing rows with unavailable forward-return labels.

In [ ]:
import numpy as np

feature_columns = tuple(bundle.manifest.feature_columns)
train_positions = split_result.indices("train")
validation_positions = split_result.indices("validation")

X_all = bundle.features.loc[:, feature_columns]
targets_all = bundle.targets
metadata_all = bundle.samples


def make_horizon_frames(horizon: int):
    target_column = f"forward_return_{horizon}d"
    if target_column not in targets_all.columns:
        raise KeyError(
            f"{target_column!r} is unavailable. "
            f"Available targets: {tuple(targets_all.columns)}"
        )

    def build(positions):
        X = X_all.iloc[positions].copy()
        y = pd.to_numeric(targets_all.iloc[positions][target_column], errors="coerce")
        metadata = metadata_all.iloc[positions].copy()

        valid = y.notna() & np.isfinite(y)
        X = X.loc[valid]
        y = y.loc[valid].astype("float64")
        metadata = metadata.loc[valid]

        frame = metadata.copy()
        frame["actual_return"] = y
        return X, y, frame

    return (*build(train_positions), *build(validation_positions))


horizon_shapes = {}
for horizon in HORIZONS:
    X_train, y_train, _, X_validation, y_validation, _ = make_horizon_frames(horizon)
    horizon_shapes[horizon] = {
        "train_rows": len(y_train),
        "validation_rows": len(y_validation),
        "feature_count": X_train.shape[1],
        "train_mean_return": y_train.mean(),
        "validation_mean_return": y_validation.mean(),
    }

pd.DataFrame(horizon_shapes).T

## 4. Notebook-local regressors

Ridge is retained as a linear negative-control baseline. XGBoost is evaluated both with native
missing-value handling and with median imputation as a sensitivity check.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor


def make_models():
    ridge = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scaler", StandardScaler()),
            ("regressor", Ridge(alpha=10.0)),
        ]
    )
    xgboost_median_imputed = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("regressor", XGBRegressor(**XGB_PARAMS)),
        ]
    )
    xgboost_native = XGBRegressor(**XGB_PARAMS)
    return {
        "ridge": ridge,
        "xgboost_native": xgboost_native,
        "xgboost_median_imputed": xgboost_median_imputed,
    }

## 5. Evaluation helpers

All daily operations go through `group_by_date` so the code remains valid whether
`trading_date` is stored as an index level or a column. Fixed top-k selections use a stable ticker
tie-break for reproducibility. Flat-score dates can be skipped to represent a no-trade decision.

In [ ]:
from scipy.stats import kendalltau, pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATE_LEVEL = "trading_date"
TICKER_LEVEL = "ticker"


def group_by_date(frame, *, sort=True):
    if DATE_LEVEL in frame.index.names:
        return frame.groupby(level=DATE_LEVEL, sort=sort)
    if DATE_LEVEL in frame.columns:
        return frame.groupby(DATE_LEVEL, sort=sort)
    raise KeyError(f"{DATE_LEVEL!r} is unavailable in the frame.")


def finite_correlation(function, x, y):
    x_array = np.asarray(x, dtype="float64")
    y_array = np.asarray(y, dtype="float64")
    mask = np.isfinite(x_array) & np.isfinite(y_array)
    if mask.sum() < 3:
        return np.nan
    if np.ptp(x_array[mask]) == 0 or np.ptp(y_array[mask]) == 0:
        return np.nan
    result = function(x_array[mask], y_array[mask])
    return float(result.statistic if hasattr(result, "statistic") else result[0])


def add_predictions(frame, predictions):
    prediction_array = np.asarray(predictions, dtype="float64").reshape(-1)
    if len(prediction_array) != len(frame):
        raise ValueError(
            "Prediction count does not match frame rows: "
            f"{len(prediction_array)} != {len(frame)}"
        )

    result = frame.copy()
    result["predicted_return"] = prediction_array
    if DATE_LEVEL not in result.index.names and DATE_LEVEL not in result.columns:
        raise KeyError(f"{DATE_LEVEL!r} is unavailable in metadata or the index.")
    return result


def aggregate_metrics(frame):
    valid = np.isfinite(frame["actual_return"]) & np.isfinite(frame["predicted_return"])
    evaluated = frame.loc[valid]
    actual = evaluated["actual_return"].to_numpy(dtype="float64")
    predicted = evaluated["predicted_return"].to_numpy(dtype="float64")
    return {
        "rows": len(evaluated),
        "mae": mean_absolute_error(actual, predicted),
        "rmse": mean_squared_error(actual, predicted) ** 0.5,
        "r2": r2_score(actual, predicted),
        "pearson": finite_correlation(pearsonr, predicted, actual),
        "spearman": finite_correlation(spearmanr, predicted, actual),
        "kendall_tau": finite_correlation(kendalltau, predicted, actual),
        "prediction_mean": predicted.mean(),
        "prediction_std": predicted.std(),
        "unique_predictions": np.unique(predicted).size,
    }


def daily_information_coefficients(frame):
    rows = []
    for signal_date, group in group_by_date(frame, sort=True):
        rows.append(
            {
                DATE_LEVEL: signal_date,
                "candidate_count": len(group),
                "spearman": finite_correlation(
                    spearmanr,
                    group["predicted_return"],
                    group["actual_return"],
                ),
                "kendall_tau": finite_correlation(
                    kendalltau,
                    group["predicted_return"],
                    group["actual_return"],
                ),
            }
        )
    return pd.DataFrame(rows)


def summarize_daily_information_coefficients(daily_ic):
    valid_spearman = daily_ic["spearman"].dropna()
    valid_kendall = daily_ic["kendall_tau"].dropna()
    return {
        "dates": len(daily_ic),
        "valid_spearman_dates": len(valid_spearman),
        "mean_daily_spearman": valid_spearman.mean(),
        "median_daily_spearman": valid_spearman.median(),
        "daily_spearman_positive_fraction": valid_spearman.gt(0).mean(),
        "valid_kendall_dates": len(valid_kendall),
        "mean_daily_kendall": valid_kendall.mean(),
        "median_daily_kendall": valid_kendall.median(),
        "daily_kendall_positive_fraction": valid_kendall.gt(0).mean(),
    }

In [ ]:
def _eligible_daily_group(group):
    valid = np.isfinite(group["predicted_return"]) & np.isfinite(group["actual_return"])
    return group.loc[valid].copy()


def _ticker_sort_values(frame):
    if TICKER_LEVEL in frame.index.names:
        return frame.index.get_level_values(TICKER_LEVEL).astype(str)
    if TICKER_LEVEL in frame.columns:
        return frame[TICKER_LEVEL].astype(str)
    return pd.Index(frame.index.astype(str))


def select_daily_top(frame, k: int, *, skip_flat_dates=False):
    if k < 1:
        raise ValueError("k must be at least 1.")

    selected_groups = []
    for _, group in group_by_date(frame, sort=True):
        eligible = _eligible_daily_group(group)
        if eligible.empty:
            continue
        if skip_flat_dates and eligible["predicted_return"].nunique() == 1:
            continue

        eligible["_ticker_sort"] = _ticker_sort_values(eligible)
        ordered = eligible.sort_values(
            ["predicted_return", "_ticker_sort"],
            ascending=[False, True],
            kind="mergesort",
        )
        selected_groups.append(ordered.head(min(k, len(ordered))).drop(columns="_ticker_sort"))

    if not selected_groups:
        return frame.iloc[0:0].copy()
    return pd.concat(selected_groups)


def select_daily_max_score_bucket(frame, *, skip_flat_dates=False):
    selected_groups = []
    for _, group in group_by_date(frame, sort=True):
        eligible = _eligible_daily_group(group)
        if eligible.empty:
            continue
        if skip_flat_dates and eligible["predicted_return"].nunique() == 1:
            continue
        maximum = eligible["predicted_return"].max()
        selected_groups.append(eligible.loc[eligible["predicted_return"].eq(maximum)])

    if not selected_groups:
        return frame.iloc[0:0].copy()
    return pd.concat(selected_groups)


def date_matched_universe_mean(frame):
    daily_means = group_by_date(frame, sort=True)["actual_return"].mean()
    return float(daily_means.mean())


def return_distribution(frame, k: int, *, skip_flat_dates=False):
    selected = select_daily_top(frame, k, skip_flat_dates=skip_flat_dates)
    returns = selected["actual_return"]
    selected_dates = (
        selected.index.get_level_values(DATE_LEVEL).nunique()
        if DATE_LEVEL in selected.index.names
        else selected[DATE_LEVEL].nunique()
    )
    full_mean = date_matched_universe_mean(frame)
    return {
        "selection": f"top_{k}",
        "skip_flat_dates": skip_flat_dates,
        "selected_dates": selected_dates,
        "selected_rows": len(selected),
        "mean_return": returns.mean(),
        "median_return": returns.median(),
        "std_return": returns.std(),
        "p05": returns.quantile(0.05),
        "p10": returns.quantile(0.10),
        "p25": returns.quantile(0.25),
        "p75": returns.quantile(0.75),
        "p90": returns.quantile(0.90),
        "p95": returns.quantile(0.95),
        "positive_fraction": returns.gt(0).mean(),
        "date_matched_universe_mean": full_mean,
        "mean_return_spread": returns.mean() - full_mean,
    }

In [ ]:
def prediction_cardinality_by_date(frame):
    rows = []
    for signal_date, group in group_by_date(frame, sort=True):
        eligible = _eligible_daily_group(group)
        if eligible.empty:
            continue
        maximum = eligible["predicted_return"].max()
        top_bucket_size = int(eligible["predicted_return"].eq(maximum).sum())
        rows.append(
            {
                DATE_LEVEL: signal_date,
                "candidate_count": len(eligible),
                "unique_predictions": eligible["predicted_return"].nunique(),
                "minimum": eligible["predicted_return"].min(),
                "maximum": maximum,
                "standard_deviation": eligible["predicted_return"].std(ddof=0),
                "top_bucket_size": top_bucket_size,
                "top_bucket_fraction": top_bucket_size / len(eligible),
                "ranking_state": (
                    "no_signal"
                    if eligible["predicted_return"].nunique() == 1
                    else "unique_top"
                    if top_bucket_size == 1
                    else "tied_top"
                ),
            }
        )
    return pd.DataFrame(rows).set_index(DATE_LEVEL)


def top_score_bucket_distribution(frame, *, skip_flat_dates=True):
    selected = select_daily_max_score_bucket(frame, skip_flat_dates=skip_flat_dates).copy()
    if selected.empty:
        return pd.Series(dtype="float64")
    return group_by_date(selected, sort=True)["actual_return"].mean()


def top_vs_bottom_score_bucket_spreads(frame):
    rows = []
    for signal_date, group in group_by_date(frame, sort=True):
        eligible = _eligible_daily_group(group)
        if eligible["predicted_return"].nunique() < 2:
            continue
        minimum = eligible["predicted_return"].min()
        maximum = eligible["predicted_return"].max()
        bottom_return = eligible.loc[
            eligible["predicted_return"].eq(minimum), "actual_return"
        ].mean()
        top_return = eligible.loc[
            eligible["predicted_return"].eq(maximum), "actual_return"
        ].mean()
        rows.append(
            {
                DATE_LEVEL: signal_date,
                "bottom_bucket_return": bottom_return,
                "top_bucket_return": top_return,
                "top_minus_bottom_spread": top_return - bottom_return,
            }
        )
    return pd.DataFrame(rows).set_index(DATE_LEVEL)

## 6. Fit every historical model/horizon combination

Progress is shown per horizon. The resulting frames are retained for all downstream diagnostics.

In [ ]:
from tqdm.auto import tqdm

model_results = {}
prediction_frames = {}
fitted_models = {}

horizon_progress = tqdm(HORIZONS, desc="Horizons")
for horizon in horizon_progress:
    horizon_progress.set_postfix_str(f"{horizon}d")
    (
        X_train,
        y_train,
        _,
        X_validation,
        _,
        validation_frame,
    ) = make_horizon_frames(horizon)

    model_items = list(make_models().items())
    for model_name, model in tqdm(
        model_items,
        desc=f"{horizon}d models",
        leave=False,
    ):
        key = (horizon, model_name)
        model.fit(X_train, y_train)
        validation_predictions = model.predict(X_validation)
        scored = add_predictions(validation_frame, validation_predictions)

        aggregate = aggregate_metrics(scored)
        daily_ic = daily_information_coefficients(scored)
        aggregate.update(summarize_daily_information_coefficients(daily_ic))

        model_results[key] = {
            **aggregate,
            "horizon": horizon,
            "model": model_name,
        }
        prediction_frames[key] = scored
        fitted_models[key] = model

aggregate_results = (
    pd.DataFrame(model_results.values())
    .set_index(["horizon", "model"])
    .sort_index()
)
aggregate_results

## 7. Fixed top-k realized-return distributions

Percentage cutoffs are omitted because a universe of roughly 119 candidates makes them redundant
with fixed top-k selections. Results are shown both across all dates and after skipping dates where
every candidate receives the same score.

In [ ]:
top_k_rows = []
for (horizon, model_name), frame in prediction_frames.items():
    for k in TOP_K_VALUES:
        for skip_flat_dates in (False, True):
            top_k_rows.append(
                {
                    "horizon": horizon,
                    "model": model_name,
                    **return_distribution(
                        frame,
                        k,
                        skip_flat_dates=skip_flat_dates,
                    ),
                }
            )

top_k_results = (
    pd.DataFrame(top_k_rows)
    .set_index(["horizon", "model", "selection", "skip_flat_dates"])
    .sort_index()
)
top_k_results

## 8. Prediction cardinality and score-bucket diagnostics

These diagnostics determine whether top-k selection represents a genuine ordering or merely an
arbitrary cut through tied predictions. `INSPECT_KEY` defaults to the 5-day native XGBoost model
because that model produced the most interesting original top-k result.

In [ ]:
INSPECT_KEY = (5, "xgboost_native")
inspected_frame = prediction_frames[INSPECT_KEY]

cardinality = prediction_cardinality_by_date(inspected_frame)
cardinality.describe(include="all")

In [ ]:
{
    "ranking_state_counts": cardinality["ranking_state"].value_counts().to_dict(),
    "top_bucket_size_counts": cardinality["top_bucket_size"].value_counts().sort_index().to_dict(),
    "most_common_prediction_values": (
        inspected_frame["predicted_return"].value_counts().head(20).to_dict()
    ),
}

In [ ]:
top_1_all_dates = select_daily_top(inspected_frame, 1, skip_flat_dates=False).copy()
top_1_all_dates["ranking_state"] = (
    top_1_all_dates.index.get_level_values(DATE_LEVEL).map(cardinality["ranking_state"])
)

ranking_state_results = top_1_all_dates.groupby("ranking_state")["actual_return"].agg(
    count="count",
    mean="mean",
    median="median",
    std="std",
    positive_fraction=lambda values: values.gt(0).mean(),
)
ranking_state_results

In [ ]:
max_score_daily_returns = top_score_bucket_distribution(
    inspected_frame,
    skip_flat_dates=True,
)
score_bucket_spreads = top_vs_bottom_score_bucket_spreads(inspected_frame)

{
    "max_score_bucket_daily_return": max_score_daily_returns.describe(
        percentiles=[0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
    ).to_dict(),
    "top_minus_bottom_score_bucket_spread": score_bucket_spreads[
        "top_minus_bottom_spread"
    ].describe(percentiles=[0.05, 0.50, 0.95]).to_dict(),
}

## 9. Optional uncertainty and matched random-ranking baseline

The original row-by-row implementation took more than 40 minutes. The helpers below avoid
repeatedly rebuilding dataframes, evaluate all fixed top-k selections in one permutation pass,
and show progress with `tqdm`.

Resampling remains disabled by default. Enable `RUN_RESAMPLING` only after choosing the model and
horizon to inspect.

In [ ]:
from tqdm.auto import tqdm

def bootstrap_top_k_means(
    frame,
    top_k_values=TOP_K_VALUES,
    *,
    iterations=BOOTSTRAP_ITERATIONS,
    seed=RANDOM_SEED,
    skip_flat_dates=True,
    show_progress=True,
):
    rng = np.random.default_rng(seed)
    selected_by_k = {
        k: select_daily_top(frame, k, skip_flat_dates=skip_flat_dates)
        for k in top_k_values
    }
    daily_means_by_k = {
        k: group_by_date(selected, sort=True)["actual_return"].mean().to_numpy()
        for k, selected in selected_by_k.items()
    }

    rows = []
    for k in tqdm(top_k_values, desc="Bootstrap top-k", disable=not show_progress):
        daily_means = daily_means_by_k[k]
        estimates = np.empty(iterations, dtype="float64")
        for iteration in range(iterations):
            sampled = rng.choice(daily_means, size=len(daily_means), replace=True)
            estimates[iteration] = sampled.mean()

        rows.append(
            {
                "selection": f"top_{k}",
                "dates": len(daily_means),
                "mean_return": daily_means.mean(),
                "bootstrap_ci_low": np.quantile(estimates, 0.025),
                "bootstrap_ci_high": np.quantile(estimates, 0.975),
            }
        )
    return pd.DataFrame(rows).set_index("selection")


def _permutation_groups(frame, *, skip_flat_dates):
    groups = []
    for _, group in group_by_date(frame, sort=True):
        eligible = _eligible_daily_group(group)
        if eligible.empty:
            continue
        if skip_flat_dates and eligible["predicted_return"].nunique() == 1:
            continue
        groups.append(
            (
                eligible["predicted_return"].to_numpy(dtype="float64"),
                eligible["actual_return"].to_numpy(dtype="float64"),
                np.asarray(_ticker_sort_values(eligible), dtype=str),
            )
        )
    return groups


def within_date_permutation_tests(
    frame,
    top_k_values=TOP_K_VALUES,
    *,
    iterations=PERMUTATION_ITERATIONS,
    seed=RANDOM_SEED,
    skip_flat_dates=True,
    show_progress=True,
):
    rng = np.random.default_rng(seed)
    groups = _permutation_groups(frame, skip_flat_dates=skip_flat_dates)
    if not groups:
        raise ValueError("No eligible date groups remain for permutation testing.")

    observed = {
        k: select_daily_top(frame, k, skip_flat_dates=skip_flat_dates)[
            "actual_return"
        ].mean()
        for k in top_k_values
    }
    null_estimates = {
        k: np.empty(iterations, dtype="float64") for k in top_k_values
    }

    iterator = tqdm(
        range(iterations),
        desc="Within-date permutations",
        disable=not show_progress,
    )
    for iteration in iterator:
        returns_by_k = {k: [] for k in top_k_values}
        for scores, actual_returns, ticker_sort in groups:
            permuted_scores = rng.permutation(scores)
            order = np.lexsort((ticker_sort, -permuted_scores))
            for k in top_k_values:
                returns_by_k[k].extend(actual_returns[order[: min(k, len(order))]])

        for k in top_k_values:
            null_estimates[k][iteration] = np.mean(returns_by_k[k])

    rows = []
    for k in top_k_values:
        null = null_estimates[k]
        observed_mean = observed[k]
        rows.append(
            {
                "selection": f"top_{k}",
                "observed_mean_return": observed_mean,
                "null_mean_return": null.mean(),
                "null_p95": np.quantile(null, 0.95),
                "one_sided_p_value": (
                    1 + np.sum(null >= observed_mean)
                )
                / (iterations + 1),
            }
        )
    return pd.DataFrame(rows).set_index("selection")

In [ ]:
if RUN_RESAMPLING:
    bootstrap_results = bootstrap_top_k_means(inspected_frame)
    permutation_results = within_date_permutation_tests(inspected_frame)
    uncertainty_results = bootstrap_results.join(permutation_results)
    display(uncertainty_results)
else:
    print(
        "Resampling skipped. Set RUN_RESAMPLING = True to run the progress-visible "
        "bootstrap and permutation tests."
    )

## 10. Visual diagnostics

These plots are descriptive only. The scatter plot reveals prediction compression, while the
score-bucket spread and daily information coefficient show whether the model generalizes beyond
a small number of top-score observations.

In [ ]:
import matplotlib.pyplot as plt

plot_frame = prediction_frames[(5, "xgboost_native")]
sample = plot_frame.sample(min(5_000, len(plot_frame)), random_state=RANDOM_SEED)

plt.figure(figsize=(9, 6))
plt.scatter(
    sample["predicted_return"],
    sample["actual_return"],
    alpha=0.2,
    s=10,
)
plt.axhline(0, linewidth=1)
plt.xlabel("Predicted 5-session return")
plt.ylabel("Actual 5-session return")
plt.title("Predicted versus actual returns — 5d XGBoost native")
plt.show()

In [ ]:
spread_series = score_bucket_spreads["top_minus_bottom_spread"]

plt.figure(figsize=(11, 5))
plt.plot(spread_series.index, spread_series, linewidth=0.8)
plt.axhline(0, linewidth=1)
plt.xlabel("Signal date")
plt.ylabel("Top-score minus bottom-score return")
plt.title("Daily score-bucket return spread — 5d XGBoost native")
plt.show()

In [ ]:
daily_ic = daily_information_coefficients(prediction_frames[(5, "xgboost_native")])

plt.figure(figsize=(11, 5))
plt.plot(daily_ic[DATE_LEVEL], daily_ic["spearman"], linewidth=0.8)
plt.axhline(0, linewidth=1)
plt.xlabel("Signal date")
plt.ylabel("Daily Spearman correlation")
plt.title("Daily cross-sectional information coefficient — 5d XGBoost native")
plt.show()

## 11. Recorded findings and handoff

The executed POC produced the following findings:

1. **Ridge is not promising.** Its validation results consistently underperformed the nonlinear
   models and the universe benchmark.
2. **XGBoost does not show broad regression or ranking quality.** Validation \(R^2\) was close to
   zero, daily rank correlations were weak or negative, and the original decile relationship was
   not monotonic.
3. **The 5-day XGBoost top scores remain interesting.** The top-1 and top-3 realized-return
   results exceeded their date-matched permutation nulls in the original 500-permutation run, but
   that evidence is provisional and should not be used for model selection by itself.
4. **The first-pass XGBoost behaves as a sparse detector rather than a smooth regressor.** Across
   roughly 119 candidates per validation date, it produced a median of four distinct scores. The
   baseline prediction `0.003708` covered 57,559 observations.
5. **The top candidate was usually distinguishable.** Among 497 validation dates, 388 had a
   unique top score, 62 had a tied top bucket of two to four stocks, and 47 gave every candidate
   the same score.
6. **Flat-score dates should be no-trade dates.** Excluding those 47 dates improved the deterministic
   top-1 mean return from approximately **1.09%** to **1.15%**, and the median from approximately
   **0.49%** to **0.87%**.
7. **The tied top bucket needs a second decision layer.** The arbitrarily selected top-1 member on
   tied dates showed unusually strong returns, but the honest unit of analysis is the complete
   tied bucket or a separate ranker applied inside it.

### Decision

This notebook establishes a useful baseline but does **not** identify a production-ready return
model. The next experiment should move to a separate notebook and compare:

- horizons 1, 2, 3, 4, and 5 sessions;
- raw forward return and forward price change measured in ATR units;
- Ridge, a regression-specific XGBRegressor baseline, and XGBRanker;
- fixed top-1, top-3, top-5, complete top-score buckets, and explicit no-trade dates;
- subperiod stability and progress-visible statistical validation.

The locked test period remains untouched.

In [ ]:
prediction_frames.keys()

In [ ]:
temp = prediction_frames[(5, "xgboost_native")]

fig, ax = plt.subplots(figsize=(10, 6))
ax.hexbin(
    x=temp["predicted_return"],
    y=temp["actual_return"],
    mincnt=1,
    bins="log",
    gridsize=300,
    cmap="terrain",
    zorder=3,    
)
ax.grid(zorder=3)
plt.show()